In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import numpy as np

# ==========================================
# 1. SETUP & DATA LOADING
# ==========================================
try:
    df = pd.read_csv('ALL_UQ_PREDICTED.csv')
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
except FileNotFoundError:
    print("Error: 'ALL_UQ_PREDICTED.csv' not found. Please upload the file.")
    # Create dummy structure to prevent crash if file missing
    df = pd.DataFrame(columns=['date', 'actual']) 
    if not df.empty: df.set_index('date', inplace=True)

# Load metrics from the Results folder
metrics_path = os.path.join('..', 'Results', 'ALL_UQ_METRICS.csv')
try:
    metrics_df = pd.read_csv(metrics_path)
    # Create a mapping of model_name to winkler_score
    winkler_map = dict(zip(metrics_df['model_name'], metrics_df['winkler_score']))
except FileNotFoundError:
    print("Warning: ALL_UQ_METRICS.csv not found. Winkler score filtering will be unavailable.")
    metrics_df = None
    winkler_map = {}

# ==========================================
# 2. COLUMN PARSING
# ==========================================
# We need to identify the "base" prediction columns vs the bounds (_L, _U)
# Base columns are those that do NOT end in _L or _U
all_cols = [c for c in df.columns if c != 'actual']
base_cols = [c for c in all_cols if not c.endswith('_L') and not c.endswith('_U')]

# Extract Metadata
models = set()
schemes = set()
seeds = set()

col_meta = {} # Map base_col -> (model, scheme, seed)

for col in base_cols:
    parts = col.split('_')
    # Assumes format: model_scheme_seed (e.g., gru_mcd_42)
    # This handles names like gru_mcd_1234 correctly
    if len(parts) >= 3:
        m, s, sd = parts[0], parts[1], parts[2]
        models.add(m)
        schemes.add(s)
        seeds.add(sd)
        col_meta[col] = (m, s, sd)

sorted_models = sorted(list(models))
sorted_schemes = sorted(list(schemes))
sorted_seeds = sorted(list(seeds), key=lambda x: int(x) if x.isdigit() else x)

# ==========================================
# 3. WIDGETS
# ==========================================
style = {'description_width': 'initial'}

w_model = widgets.Dropdown(options=['All'] + sorted_models, value='All', description='Model:', style=style)
w_scheme = widgets.Dropdown(options=['All'] + sorted_schemes, value='All', description='UQ Scheme:', style=style)
w_seed = widgets.Dropdown(options=['All'] + sorted_seeds, value='All', description='Seed:', style=style)
w_show_band = widgets.Checkbox(value=True, description='Show Uncertainty Bands')
w_show_coverage = widgets.Checkbox(value=False, description='Show Band Coverage (Green=Inside, Red=Outside)')
w_top_winkler = widgets.Checkbox(value=False, description='Filter Top 10% Smallest Winkler Score (if available)')
w_show_rolling = widgets.Checkbox(value=False, description='Show Rolling PICP & n-MPIW (30-day window)')
w_show_events = widgets.Checkbox(value=False, description='Show Geopolitical Events')

# Date Slider
dates = df.index
if len(dates) > 0:
    w_range = widgets.IntRangeSlider(
        value=[0, len(dates)-1],
        min=0, max=len(dates)-1, step=1,
        description='Date Zoom:',
        continuous_update=False,
        layout=widgets.Layout(width='95%')
    )
else:
    w_range = widgets.IntRangeSlider(min=0, max=1)

# ==========================================
# 4. PLOTTING LOGIC
# ==========================================
def plot_uq_series(model, scheme, seed, idx_range, show_band, show_coverage, top_winkler, show_rolling, show_events):
    if df.empty:
        print("No data available.")
        return

    # Filter based on top 10% smallest winkler score if enabled
    cols_to_plot_final = list(base_cols)
    
    if top_winkler and metrics_df is not None:
        # Calculate top 10% threshold for smallest winkler scores
        winkler_threshold = metrics_df['winkler_score'].quantile(0.1)
        top_10_models = metrics_df[metrics_df['winkler_score'] <= winkler_threshold]['model_name'].unique()
        
        # Filter base_cols to only include those in top 10%
        cols_to_plot_final = [col for col in base_cols if col in top_10_models]
        if not cols_to_plot_final:
            print(f"No models in top 10% smallest winkler score (threshold: {winkler_threshold:.4f})")
    elif top_winkler and metrics_df is None:
        print("Winkler score metrics not available. Cannot filter.")

    start_idx, end_idx = idx_range
    sub_df = df.iloc[start_idx : end_idx+1]
    
    # Filter columns to plot based on dropdown selection
    cols_to_plot = []
    for col in cols_to_plot_final:
        m, s, sd = col_meta.get(col, (None, None, None))
        
        if model != 'All' and m != model: continue
        if scheme != 'All' and s != scheme: continue
        if seed != 'All' and sd != seed: continue
        
        cols_to_plot.append(col)
    
    # --- PLOT ---
    if show_rolling and len(cols_to_plot) > 0:
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 12), gridspec_kw={'height_ratios': [2, 1]})
    else:
        fig, ax1 = plt.subplots(figsize=(18, 8))
    
    # 1. Plot Actual Data (Black Line)
    ax1.plot(sub_df.index, sub_df['actual'], 
             label='Actual Data', color='black', linewidth=2.5, alpha=0.9, zorder=100)
    
    # 2. Plot Models & Bands
    # Limit colors if too many lines are selected
    colors = sns.color_palette("bright", len(cols_to_plot))
    
    if not cols_to_plot:
        ax1.text(0.5, 0.5, "No models match these filters", ha='center', transform=ax1.transAxes, fontsize=14)
    else:
        for i, col in enumerate(cols_to_plot):
            # Get winkler score for this model
            winkler_score = winkler_map.get(col, None)
            if winkler_score is not None:
                label = f"{col} (W: {winkler_score:.4f})"
            else:
                label = col
            
            # Plot the Base Prediction Line
            ax1.plot(sub_df.index, sub_df[col], label=label, color=colors[i], linewidth=2, alpha=1.0)
            
            # Plot the Uncertainty Band
            if show_band:
                col_L = f"{col}_L"
                col_U = f"{col}_U"
                
                # Verify bounds exist in the dataframe before plotting
                if col_L in sub_df.columns and col_U in sub_df.columns:
                    ax1.fill_between(sub_df.index, 
                                     sub_df[col_L], 
                                     sub_df[col_U], 
                                     color=colors[i], 
                                     alpha=0.2, # Transparency for the band
                                     label='_nolegend_') # Exclude band from legend
                    
                    # Plot Band Coverage Markers
                    if show_coverage:
                        col_L = f"{col}_L"
                        col_U = f"{col}_U"
                        
                        # Check which actual points fall inside/outside the band
                        actual_vals = sub_df['actual']
                        lower_bounds = sub_df[col_L]
                        upper_bounds = sub_df[col_U]
                        
                        # Points inside the band (limegreen bullets)
                        inside_mask = (actual_vals >= lower_bounds) & (actual_vals <= upper_bounds)
                        if inside_mask.any():
                            ax1.plot(sub_df.index[inside_mask], actual_vals[inside_mask], 
                                    marker='o', markersize=6, color='limegreen', linestyle='None', 
                                    alpha=0.7, zorder=200)
                        
                        # Points outside the band (red X)
                        outside_mask = (actual_vals < lower_bounds) | (actual_vals > upper_bounds)
                        if outside_mask.any():
                            ax1.plot(sub_df.index[outside_mask], actual_vals[outside_mask], 
                                    marker='x', markersize=8, color='red', linestyle='None', 
                                    alpha=0.9, zorder=200)
            
            # Plot Rolling PICP and n-MPIW
            if show_rolling:
                col_L = f"{col}_L"
                col_U = f"{col}_U"
                
                if col_L in df.columns and col_U in df.columns:
                    # Find where valid data starts for this model (first non-NaN prediction)
                    pred_col = col
                    first_valid_idx = df[pred_col].first_valid_index()
                    
                    if first_valid_idx is not None:
                        # Only calculate rolling metrics from the first valid data point
                        valid_mask = df.index >= first_valid_idx
                        df_valid = df[valid_mask].copy()
                        
                        # Calculate PICP: is actual within bounds?
                        actual_vals = df_valid['actual']
                        lower_bounds = df_valid[col_L]
                        upper_bounds = df_valid[col_U]
                        
                        coverage = ((actual_vals >= lower_bounds) & (actual_vals <= upper_bounds)).astype(int)
                        rolling_picp = coverage.rolling(window=30).mean()
                        
                        # Calculate n-MPIW: interval width normalized by actual range
                        interval_width = upper_bounds - lower_bounds
                        rolling_mpiw = interval_width.rolling(window=30).mean()
                        actual_range = actual_vals.max() - actual_vals.min()
                        if actual_range > 0:
                            rolling_nmpiw = rolling_mpiw / actual_range
                        else:
                            rolling_nmpiw = rolling_mpiw
                        
                        # Create dual axes if not already done
                        if i == 0:
                            ax2_picp = ax2
                            ax2_nmpiw = ax2.twinx()
                            # Add CI threshold lines once
                            ax2_picp.axhline(y=0.95, color='green', linestyle='--', linewidth=2, alpha=0.7, label='95% CI')
                            ax2_picp.axhline(y=0.90, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='90% CI')
                            ax2_picp.set_ylabel('PICP (Coverage Probability)', color='black', fontsize=11)
                            ax2_nmpiw.set_ylabel('n-MPIW (Normalized Width)', color='black', fontsize=11)
                            ax2_picp.tick_params(axis='y', labelcolor='black')
                            ax2_nmpiw.tick_params(axis='y', labelcolor='black')
                            ax2_picp.legend(loc='upper left', fontsize=9)
                        
                        # Darken color for PICP and lighten for n-MPIW
                        picp_color = tuple(c * 0.8 for c in colors[i][:3]) if isinstance(colors[i], tuple) else colors[i]
                        nmpiw_color = tuple(min(c + 0.2, 1.0) for c in colors[i][:3]) if isinstance(colors[i], tuple) else colors[i]
                        
                        ax2_picp.plot(rolling_picp.index, rolling_picp, label=f"{col} PICP", 
                                      color=picp_color, linewidth=2.5, linestyle='-', alpha=0.85)
                        ax2_nmpiw.plot(rolling_nmpiw.index, rolling_nmpiw, label=f"{col} n-MPIW", 
                                       color=nmpiw_color, linewidth=2.5, linestyle='--', alpha=0.85)

    # Plot Geopolitical Events
    if show_events:
        y_min, y_max = ax1.get_ylim()
        y_range = y_max - y_min
        
        # Event 1: US Credit Downgrade (2023-08-01)
        event_date_1 = pd.to_datetime('2023-08-01')
        if sub_df.index[0] <= event_date_1 <= sub_df.index[-1]:
            ax1.axvline(x=event_date_1, color='#FF6B6B', linestyle='--', linewidth=2, alpha=0.7, zorder=50)
            ax1.text(event_date_1, y_max - (y_range * 0.05), 'US Credit Downgrade',
                    rotation=0, verticalalignment='bottom', horizontalalignment='center',
                    fontsize=9, color='#FF6B6B', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#FF6B6B'))
        
        # Event 2: Pengumuman Pemilu KPU RI (2024-03-20)
        event_date_2 = pd.to_datetime('2024-03-20')
        if sub_df.index[0] <= event_date_2 <= sub_df.index[-1]:
            ax1.axvline(x=event_date_2, color='#4ECDC4', linestyle='--', linewidth=2, alpha=0.7, zorder=50)
            ax1.text(event_date_2, y_max - (y_range * 0.1), 'Pengumuman Pemilu RI',
                    rotation=0, verticalalignment='bottom', horizontalalignment='center',
                    fontsize=9, color='#4ECDC4', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#4ECDC4'))
        
        # Event 3: Middle-East Escalation (2024-04-13)
        event_date_3 = pd.to_datetime('2024-04-13')
        if sub_df.index[0] <= event_date_3 <= sub_df.index[-1]:
            ax1.axvline(x=event_date_3, color='#FF8C42', linestyle='--', linewidth=2, alpha=0.7, zorder=50)
            ax1.text(event_date_3, y_max - (y_range * 0.05), 'Middle-East Escalation',
                    rotation=0, verticalalignment='bottom', horizontalalignment='center',
                    fontsize=9, color='#FF8C42', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#FF8C42'))
        
        # Event 4: BI Rate turun 0.25 bps (2024-09-18)
        event_date_4 = pd.to_datetime('2024-09-18')
        if sub_df.index[0] <= event_date_4 <= sub_df.index[-1]:
            ax1.axvline(x=event_date_4, color='#95E1D3', linestyle='--', linewidth=2, alpha=0.7, zorder=50)
            ax1.text(event_date_4, y_max - (y_range * 0.05), 'BI Rate turun 0.25 bps',
                    rotation=0, verticalalignment='bottom', horizontalalignment='center',
                    fontsize=9, color='#95E1D3', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#95E1D3'))
        
        # Event 5: US Election, Trump Wins (2024-11-05)
        event_date_5 = pd.to_datetime('2024-11-05')
        if sub_df.index[0] <= event_date_5 <= sub_df.index[-1]:
            ax1.axvline(x=event_date_5, color='#F38181', linestyle='--', linewidth=2, alpha=0.7, zorder=50)
            ax1.text(event_date_5, y_max - (y_range * 0.1), 'US Election, Trump Wins',
                    rotation=0, verticalalignment='bottom', horizontalalignment='center',
                    fontsize=9, color='#F38181', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='#F38181'))

    ax1.set_title(f"Uncertainty Quantification Analysis: {len(cols_to_plot)} Models Selected", fontsize=16)
    ax1.set_ylabel('Prediction Value')
    ax1.set_xlabel('Date')
    ax1.grid(True, linestyle='--', alpha=0.5)
    
    # Smart Legend: Hide if too many items
    if len(cols_to_plot) <= 8:
        ax1.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    else:
        ax1.text(1.02, 0.95, f"Legend hidden\n({len(cols_to_plot)} models selected)\nFilter to < 8 to see details.", 
                 transform=ax1.transAxes, fontsize=11, verticalalignment='top')

    if show_rolling and len(cols_to_plot) > 0:
        ax2.set_xlabel('Date')
        ax2.grid(True, linestyle='--', alpha=0.5)
        ax2.set_title('Rolling 30-Day Metrics (PICP and n-MPIW)', fontsize=14)

    plt.tight_layout()
    plt.show()

# ==========================================
# 5. LAYOUT DISPLAY
# ==========================================
ui = widgets.VBox([
    widgets.HBox([w_model, w_scheme, w_seed, w_show_band]),
    widgets.HBox([w_show_coverage, w_top_winkler]),
    widgets.HBox([w_show_rolling, w_show_events]),
    w_range
])

out = widgets.interactive_output(plot_uq_series, {
    'model': w_model, 
    'scheme': w_scheme, 
    'seed': w_seed,
    'idx_range': w_range,
    'show_band': w_show_band,
    'show_coverage': w_show_coverage,
    'top_winkler': w_top_winkler,
    'show_rolling': w_show_rolling,
    'show_events': w_show_events
})

display(ui, out)

Output()